In [ ]:
#Spark connection
import os
import socket
from pyspark.sql import SparkSession
 
APACHE_MASTER_IP = socket.gethostbyname("apache-spark-master-0.apache-spark-headless.apache-spark.svc.cluster.local")
APACHE_MASTER_URL = f"spark://{APACHE_MASTER_IP}:7077"
POD_IP = os.environ["MY_POD_IP"]
SPARK_APP_NAME = f"spark-{os.environ['HOSTNAME']}"
JARS = "/nfs/env/lib/python3.8/site-packages/pyspark/jars/clickhouse-native-jdbc-shaded-2.6.5.jar"
MEM = "512m"
CORES = 1
 
spark = SparkSession.\
        builder.\
        appName(SPARK_APP_NAME).\
        master(APACHE_MASTER_URL).\
        config("spark.executor.memory", MEM).\
        config("spark.jars", JARS).\
        config("spark.executor.cores", CORES).\
        config("spark.driver.host", POD_IP).\
        getOrCreate()

In [ ]:
import pandas as pd

pd_orders = pd.read_csv('orders.csv')  

In [ ]:
df_orders = spark.createDataFrame(
    (pd_orders), 
    schema = 'order_id INT, order_date STRING, order_customer_id INT, order_status STRING'
)

df_orders.printSchema()

In [ ]:
from pyspark.sql.functions import date_format

df_orders.show()

In [ ]:
df_orders.count()

In [ ]:
#Знакомимся с aggregate function count
#Группируем число заказов по статусам
#Обратите внимание как мы сразу можем отсортировать результат
#Обратите внимание на alias('total'). Что будет если мы его уберем? Какое имя колонки нужно будет задать в orderBy?

from pyspark.sql.functions import count

df_agg = df_orders.groupBy('order_status').agg(count("*").alias('total')).orderBy('total', ascending=False)

df_agg.show()

In [ ]:
#Находим минимальное значение order_id и order_customer_id на каждую order_date

df_orders. \
    groupBy('order_date'). \
    min(). \
    orderBy('order_date'). \
    show()

In [ ]:
pd_order_items = pd.read_csv('order_items.csv') 

df_order_items = spark.createDataFrame(
    (pd_order_items),
    schema = "id BIGINT, \
    order_id BIGINT, \
    product_id BIGINT, \
    quantity BIGINT, \
    subtotal DOUBLE, \
    product_price DOUBLE \
    "
)

df_order_items.show(5)

In [ ]:
#Считаем для каждого order_id число проданных товаров и общую сумму
#Обратите внимание как мы переименовали колонки с помощью toDF, попробуйте убрать toDF и проверьте как выглядит DF

df_order_items. \
    select('order_id', 'quantity', 'subtotal'). \
    groupBy('order_id'). \
    sum('quantity', 'subtotal'). \
    orderBy('sum(subtotal)', ascending=False). \
    toDF('order_id', 'order_quantity', 'order_revenue').show()

In [ ]:
#Пример использования функции round и замены колонки с помощью withColumn
#Попробуйте в withColumn подставить другое имя колонки.
#Например: 
#withColumn('order_revenue_round', round('order_revenue', 2)). \

from pyspark.sql.functions import round

df_order_items. \
    groupBy('order_id'). \
    sum('quantity', 'subtotal'). \
    toDF('order_id', 'order_quantity', 'order_revenue'). \
    withColumn('order_revenue', round('order_revenue', 2)). \
    show()

In [ ]:
#Тот же результат с использованием другого синтаксиса
from pyspark.sql.functions import round, sum, avg, max, count

df_order_items. \
    groupBy('order_id'). \
    agg(sum('quantity').alias('order_quanity'), round(sum('subtotal'), 2).alias('order_revenue')). \
    show()

In [ ]:
#Пример с использованием различных агрегирующих функций
from pyspark.sql.functions import round, sum, avg, max, min

df_order_items. \
    groupBy('order_id'). \
    agg(
        sum('quantity').alias('order_quanity'), 
        min('quantity').alias('min_order_quantity'),
        round(sum('subtotal'), 2).alias('order_revenue'),
        min('subtotal').alias('min_order_item_subtotal'),
        max('product_price').alias('max_product_price'),
        round(avg('product_price'), 2).alias('avg_product_price')
    ).show()

In [ ]:
#Добавляем фильтр и count
from pyspark.sql.functions import round, sum, avg, max, min, count

df_order_items. \
    where('order_id = 5'). \
    groupBy('order_id'). \
    agg(
        count('*').alias('count'),
        sum('quantity').alias('order_quanity'), 
        min('quantity').alias('min_order_quantity'),
        round(sum('subtotal'), 2).alias('order_revenue'),
        min('subtotal').alias('min_order_item_subtotal'),
        max('product_price').alias('max_product_price'),
        round(avg('product_price'), 2).alias('avg_product_price')
    ).show()

In [ ]:
#Дополнительные примеры
# Создадим sample DF
sampleData = [("Alex","Backend","MSK",190000,34,100000),
    ("Vladimir","Backend","MSK",186000,27,200000),
    ("Egor","Backend","VRN",181000,30,230000),
    ("Maria","Frontend","VRN",290000,24,230000),
    ("Roman","Frontend","EKT",199000,40,240000),
    ("Anton","Frontend","EKT",283000,36,190000),
    ("Vika","ML","SPB",179000,53,150000),
    ("Dasha","ML","VRN",180000,25,180000),
    ("Peter","ML","SPB",191000,50,21000)
  ]

schema = ["employee_name","department","state","salary","age","bonus"]
df = spark.createDataFrame(data=sampleData, schema = schema)
df.printSchema()
df.show(truncate=False)

In [ ]:
from pyspark.sql.functions import sum, avg, max, count, col

#groupby agg
df.groupBy("department") \
    .agg(count("*").alias("count")
     ) \
    .show(truncate=False)

#groupby нескольких колонок
df.groupBy("department","state") \
    .agg(count("*").alias("count")
     ) \
    .show(truncate=False)

In [ ]:
#Применяем несколько агрегаций
df.groupBy("department") \
    .agg(sum("salary").alias("sum_salary"), \
         avg("salary").alias("avg_salary"), \
         sum("bonus").alias("sum_bonus"), \
         max("bonus").alias("max_bonus") \
     ) \
    .show(truncate=False)

In [ ]:
#Используем where
df.groupBy("department") \
    .agg(sum("salary").alias("sum_salary"), \
      avg("salary").alias("avg_salary"), \
      sum("bonus").alias("sum_bonus"), \
      max("bonus").alias("max_bonus")) \
    .where(col("sum_bonus") >= 200000) \
    .show(truncate=False)

In [ ]:
#Используем SQL
df.createOrReplaceTempView("EMP")

sql_str="select department, sum(salary) as sum_salary," \
"avg(salary) as avg_salary," \
"sum(bonus) as sum_bonus," \
"max(bonus) as max_bonus" \
" from EMP "  \
" group by department having sum_bonus >= 200000"
spark.sql(sql_str).show()

In [ ]:
spark.stop()